# Readers

> Read spectra from instrument files into a `Spectra`

In [ ]:
#| default_exp readers

In [ ]:
#| export
from fastcore.all import *
from pathlib import Path
import numpy as np, re, struct
from soilspecdata.types import *

In [ ]:
import matplotlib.pyplot as plt

## Thermo OMNIC files

In [ ]:
#| export
_spa_ytypes = {17: 'abs'} # OMNIC y-unit codes that `read_spa` supports, mapped to `Spectra.measurement_type`

def read_spa(
    fn, # Path to a Thermo OMNIC `.spa` file
)->tuple: # (wavenumbers in increasing order, values, measurement type)
    "Read the spectrum in a Thermo OMNIC `.spa` file"
    b = Path(fn).read_bytes()
    blocks = {}
    for pos in range(288, len(b) - 16, 16):
        if b[pos] == 0: break
        blocks.setdefault(b[pos], struct.unpack('<II', b[pos+2:pos+10]))
    hdr, (off, _) = blocks[2][0], blocks[3]
    n, = struct.unpack('<I', b[hdr+4:hdr+8])
    xcode, ycode = b[hdr+8], b[hdr+12]
    if xcode != 1: raise ValueError(f"x-unit code {xcode} is not cm⁻¹")
    if ycode not in _spa_ytypes: raise ValueError(f"y-unit code {ycode} is not supported, only {list(_spa_ytypes)}")
    x0, x1 = struct.unpack('<ff', b[hdr+16:hdr+24])
    wns, vals = np.linspace(x0, x1, n), np.frombuffer(b, '<f4', count=n, offset=off).astype(float)
    if x0 > x1: wns, vals = wns[::-1].copy(), vals[::-1].copy()
    return wns, vals, _spa_ytypes[ycode]

Thermo Nicolet spectrometers save each spectrum as an OMNIC `.spa` file. `read_spa` reads one file and returns its wavenumbers, its values and its measurement type. The wavenumbers are in increasing order, as `Spectra` expects. The file stores only the first and last wavenumbers and the number of points, so `read_spa` rebuilds the evenly spaced grid from them.

`read_spa` reads the unit codes in the file header. It accepts wavenumbers in cm⁻¹ and absorbance values, and raises a `ValueError` for other units. The background files of the Fukushima measurements, for example, carry y-unit code 15, not absorbance.

This file is replicate `a` of Fukushima sample 1160, measured by Jumpei. Its 1,738 points cover 650 to 4000 cm⁻¹ at the instrument's native step of about 1.93 cm⁻¹. The CO₂ band from 2266 to 2391 cm⁻¹ is still present:

In [ ]:
wns_spa, spc, mtype = read_spa('files/1160a.SPA')
test_eq(mtype, 'abs')
test_eq(len(wns_spa), 1738)
test_close(wns_spa[[0, -1]], [649.893, 3999.641], eps=1e-3)
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(wns_spa, spc)
ax.invert_xaxis()
ax.set_xlabel('Wavenumber (cm⁻¹)');

The grid depends on the measurement settings, so different batches can differ. Sample 100, measured by Ayane, starts at 600 cm⁻¹, and its grid has 1,764 points with the same step. Spectra from several batches must be interpolated onto a shared grid before they go into one `Spectra`:

In [ ]:
wns_ayane, _, _ = read_spa('files/100a.SPA')
test_eq(len(wns_ayane), 1764)
wns_ayane[[0, -1]], np.diff(wns_ayane).mean(), np.diff(wns_spa).mean()

In [ ]:
#| export
def load_spa(
    fns, # Paths of `.spa` files, with one file per replicate
    wns, # Wavenumbers to interpolate onto, in cm⁻¹, in increasing order
    id_re=r'^\d+', # Regex whose match at the start of a file name is the sample ID
)->Spectra: # One spectrum per sample, the mean of its replicates
    "Read `.spa` files, interpolate each onto `wns`, and average the replicates of each sample"
    wns, groups = np.asarray(wns), {}
    for fn in map(Path, fns):
        m = re.match(id_re, fn.name)
        if not m: raise ValueError(f"No sample ID in {fn.name}")
        x, vals, mtype = read_spa(fn)
        if wns[0] < x[0] or wns[-1] > x[-1]: raise ValueError(f"{fn.name} covers {x[0]:.2f}–{x[-1]:.2f} cm⁻¹, which does not include `wns`")
        groups.setdefault(m[0], []).append(np.interp(wns, x, vals))
    return Spectra(wavenumbers=wns, spectra=np.stack([np.mean(o, axis=0) for o in groups.values()]),
        measurement_type=mtype, sample_ids=np.array(list(groups)))

`load_spa` builds a `Spectra` from many `.spa` files. Files whose names start with the same sample ID are replicates of one sample, wherever they are stored. It interpolates each file onto `wns` with linear interpolation, then averages the replicates. Replicates of one sample usually share a grid, and linear interpolation then gives the same result as averaging first. Interpolating first also handles replicates measured on different grids.

Here the two replicates of sample 1160 and the replicate of sample 100 go onto the grid shared with OSSL, from 650 to 3998 cm⁻¹ at a 2 cm⁻¹ step. The sample IDs are in order of first appearance:

In [ ]:
grid = np.arange(650, 3999, 2)
fuku = load_spa(['files/1160a.SPA', 'files/1160b.SPA', 'files/100a.SPA'], grid)
reps = [np.interp(grid, *read_spa(f'files/1160{r}.SPA')[:2]) for r in 'ab']
test_eq(fuku.sample_ids, ['1160', '100'])
test_close(fuku.spectra[0], np.mean(reps, axis=0))
fuku.spectra.shape, fuku.measurement_type

`load_spa` does not extrapolate. The Fukushima files end at 3999.64 cm⁻¹, so a grid that ends at 4000 cm⁻¹, as OSSL does, raises a `ValueError`:

In [ ]:
expect_fail(lambda: load_spa(['files/1160a.SPA'], np.arange(650, 4001, 2)), contains='does not include `wns`')